# Improving Predictive Analytics: Baseline vs Random Forest

## 1. Diagnosis of Baseline Weaknesses
The current baseline model uses `LogisticRegression` with StandardScaler. While interpretable and fast, linear models have strict mathematical assumptions that limit their predictive capability on complex tabular data:
- **Linear Decision Boundaries**: Logistic Regression assumes the log-odds of the target (client subscribing) are a linear combination of the features. It cannot inherently capture non-linear relationships or complex feature interactions (like age * job type or campaign duration * exact day) unless we explicitly engineer them, which is extremely difficult.
- **Sensitivity to Outliers**: Even with standard scaling, numerical features with heavy tails or severe outliers can pull the hyper-plane aggressively in an attempt to minimize the loss function.
- **Severe Class Imbalance**: We have a rare positive class (`yes`). While `class_weight='balanced'` adjusts the intercept, a single linear boundary may still forcefully sacrifice local pockets of predicting 'yes' in feature space due to being overpowered globally by 'no'.

## 2. Proposed Improvements
Here are 3 ways we could improve over the baseline, given its weaknesses:
1. **Ensemble Tree Models (e.g. Random Forest, XGBoost, LightGBM)**: Tree-based ensemble algorithms construct a multitude of decision trees. They naturally learn highly non-linear decision boundaries, automatically model feature interactions, and are largely insensitive to outliers since they split on feature thresholds rather than calculating distances.
2. **Advanced Feature Engineering + Logistic Regression with L1 Regularization (Lasso)**: We could manually generate polynomial features and interaction terms (e.g., degree 2 or 3) and then apply L1 regularization to perform aggressive feature selection, forcing the Logistic Regression to consider complex combinations.
3. **SMOTE (Synthetic Minority Over-sampling Technique) + Threshold Tuning**: Since the data is imbalanced, rather than just adjusting class weights, we could create synthetic positive examples (SMOTE) to create a denser feature space for 'yes' customers, followed by tuning the probability threshold to maximise average precision.

## 3. Final Chosen Approach
**Selected Approach:** We will implement an ensemble tree model: **Random Forest Classifier**.

**Justification**:
Switching to an ensemble tree model is the most robust and commonly successful way to improve significantly upon a linear tabular baseline without over-engineering complex feature generation or dealing with the extensive mathematical artefacts of distance-calculating algorithms. We use a Random Forest as it natively handles non-linearities, is highly resistant to over-fitting compared to individual trees, easily incorporates `class_weight='balanced'`, and does not require complex tuning to beat a linear baseline immediately.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# --- Data Loading & Leakage Prevention (Identical to Baseline) ---
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Strict Leakage Handling - Must drop duration as it is unknown until after the target occurs
df = df.drop(columns=['duration'])

# Mathematical Artefact Handling (pdays=999 implies not contacted)
df['contacted_previously'] = (df['pdays'] != 999).astype(int)
df = df.drop(columns=['pdays'])

df['y'] = df['y'].map({'no': 0, 'yes': 1})

# Feature definition
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(exclude=['object', 'datetime']).columns.tolist()
if 'y' in numerical_features: numerical_features.remove('y')
if 'contacted_previously' in numerical_features: numerical_features.remove('contacted_previously')
numerical_features.append('contacted_previously')

X = df.drop(columns=['y'])
y = df['y']

# Fair Evaluation Protocol: Identical Reproducible Splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [2]:
# --- Identical Preprocessor Definition ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categorical_features)
    ])

# --- Model 1: The Original Baseline ---
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# --- Model 2: The Improved Approach (Random Forest) ---
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, 
        max_depth=10,        # Controlled depth to prevent over-fitting
        min_samples_leaf=4,  # Regularisation approach
        random_state=42, 
        class_weight='balanced',
        n_jobs=-1
    ))
])

print("Training models...")
baseline_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
print("Training complete.")

Training models...


Training complete.


In [3]:
# --- 5. Fair Comparison & Evaluation ---
def evaluate_model(name, pipeline):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n{'='*15} {name} {'='*15}")
    print(classification_report(y_test, y_pred))
    print(f"Test ROC-AUC: {roc_auc:.4f}")
    print(f"Test PR-AUC:  {pr_auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    return roc_auc, pr_auc

print("--- EVALUATION ON HOLD-OUT TEST SET ---")
roc_baseline, pr_baseline = evaluate_model("Baseline (Logistic Regression)", baseline_model)
roc_rf, pr_rf = evaluate_model("Improved Model (Random Forest)", rf_model)

summary = pd.DataFrame({
    'Model': ['LogReg Baseline', 'Random Forest Improved'],
    'ROC-AUC': [roc_baseline, roc_rf],
    'PR-AUC (Avg Precision)': [pr_baseline, pr_rf]
})

import IPython.display as display
print("\n--- RESULT SUMMARY TABLE ---")
display.display(summary)

--- EVALUATION ON HOLD-OUT TEST SET ---

=============== Baseline (Logistic Regression) ===============
              precision    recall  f1-score   support

           0       0.95      0.86      0.90      7310
           1       0.37      0.65      0.47       928

    accuracy                           0.84      8238
   macro avg       0.66      0.75      0.69      8238
weighted avg       0.88      0.84      0.85      8238

Test ROC-AUC: 0.8010
Test PR-AUC:  0.4601
Confusion Matrix:
[[6280 1030]
 [ 329  599]]



=============== Improved Model (Random Forest) ===============
              precision    recall  f1-score   support

           0       0.95      0.88      0.92      7310
           1       0.41      0.64      0.50       928

    accuracy                           0.86      8238
   macro avg       0.68      0.76      0.71      8238
weighted avg       0.89      0.86      0.87      8238

Test ROC-AUC: 0.8128
Test PR-AUC:  0.4912
Confusion Matrix:
[[6468  842]
 [ 333  595]]

--- RESULT SUMMARY TABLE ---


,Model,ROC-AUC,PR-AUC (Avg Precision)
0,LogReg Baseline,0.800989,0.460054
1,Random Forest Improved,0.812777,0.491239


## 6. Why the Improvement Helps
The **Random Forest** provides superior performance because it naturally constructs step-functions that cleanly box in regions of the decision space, which is critical when rules like `age > 60` and `job == retired` interact heavily to predict a positive subscription rate. Logistic regression forces identical age-slopes regardless of job type. 
By setting `max_depth=10`, the Random Forest avoids splitting all the way down to pure individual leaves, meaning it captures powerful non-linear macro-trends in the feature space without over-adapting to microscopic noise (high variance).

## 7. Risk Checks (Validity & Sanity)
- **Overfitting**: Complex tree models are prone to overfitting. We mitigated this by setting `max_depth=10`, enforcing `min_samples_leaf=4`, and capping trees at `200`. 
- **Leakage Check**: The highly predictive `duration` column is categorically dropped, enforcing adherence to the project specification that predictions occur *before* the call happens.
- **Fair Evaluation**: Identical fixed seed splits (`random_state=42`), identical strict stratification arrays (`stratify=y`), and exactly identical underlying pre-processors (StandardScaler and OneHotEncoder) were used across both pipelines.
- **Business Validity**: `pdays=999` is processed out into a boolean indicator `contacted_previously` to prevent distance-bias artefacts in model calculations.